In [30]:
import numpy as np

wf = np.load('generated_wf.npy')
print(wf.shape)
print(wf.dtype)
print(wf[1])

(2048,)
complex64
(0.06705877-0.005113327j)


In [31]:
import torch
import torch.nn as nn

def count_model_params(model: nn.Module) -> dict:
    """Computes trainable, non-trainable, and total parameters in PyTorch."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return {
        "trainable": trainable,
        "non_trainable": non_trainable,
        "total": trainable + non_trainable,
    }

def get_conv1d_params(in_length: int, out_length: int, in_channels: int, out_channels: int, bias: bool = True):
    """
    Finds valid (kernel_size, stride, padding) configurations to map in_length -> out_length,
    along with total learnable parameters and an instantiated nn.Conv1d layer.
    """
    candidates = []
    for stride in [1, 2, 4, 8]:
        for kernel_size in range(2, 17):
            for padding in range(kernel_size):
                num = in_length + 2 * padding - kernel_size
                if num >= 0 and num % stride == 0:
                    l_out = num // stride + 1
                    if l_out == out_length:
                        num_params = out_channels * in_channels * kernel_size + (out_channels if bias else 0)
                        candidates.append({
                            "kernel_size": kernel_size,
                            "stride": stride,
                            "padding": padding,
                            "trainable_params": num_params,
                            "layer": nn.Conv1d(
                                in_channels=in_channels,
                                out_channels=out_channels,
                                kernel_size=kernel_size,
                                stride=stride,
                                padding=padding,
                                bias=bias,
                            ),
                        })
    return candidates


In [ ]:

class Encoder1(nn.Module):
    def __init__(self, negative_slope: float = 0.2):
        super().__init__()
        self.co1 = nn.Conv1d(in_channels=1, out_channels= 4)
        self.bn2 = nn.BatchNorm1d(512)
        self.co1 = nn.Conv1d(in_channels=1, out_channels=4, kernel_size=2, stride=2, padding=0, bias = False)#out_L = 256
        self.bn3 = nn.BatchNorm1d(4)
        self.co2 = nn.Conv1d(in_channels=4, out_channels=1, kernel_size=2, stride=2, padding=0, bias= False)# out_L = 128
        self.bn4 = nn.BatchNorm1d(1)
        self.fc5 = nn.Linear(128, 64)
        self.act = nn.LeakyReLU(negative_slope=negative_slope)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.bn1(self.fc1(x)))
        x = self.act(self.bn2(self.fc2(x)))
        x = x.unsqueeze(1)
        x = self.act(self.bn3(self.co1(x)))
        x = self.act(self.bn4(self.co2(x)))
        x = x.flatten(start_dim=1)
        return self.fc5(x)

class Decoder1(nn.Module):
    def __init__(self, negative_slope: float = 0.2):
        super().__init__()
        self.fc1 = nn.Linear(64, 128, bias=False)
        self.bn1 = nn.BatchNorm1d(128)
        self.co1 = nn.Conv1d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1, bias = False)#out_L = 128
        self.bn2 = nn.BatchNorm1d(8)
        self.fc2 = nn.Linear(1024,2048, bias = False)
        self.act = nn.LeakyReLU(negative_slope=negative_slope)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.bn1(self.fc1(x)))
        x = x.unsqueeze(1)
        x = self.act(self.bn2(self.co1(x)))
        x = x.flatten(start_dim = 1)
        return self.fc2(x)

class AutoEncoderG(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder1()
        self.decoder = Decoder1()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


model1 = AutoEncoderG()
model1.eval()
dummy_params = torch.randn(1, 3)
output = model1(dummy_params)
print("Output shape:", output.shape)
print("Parameters:", count_model_params(model1))


Output shape: torch.Size([1, 2048])
Parameters: {'trainable': 2181122, 'non_trainable': 0, 'total': 2181122}


In [33]:
model2 = AutoEncoderTasos()
dummy_params = torch.randn(1,3)
model2.eval()
output = model2(dummy_params)
print("Output shape:", output.shape)
print("Parameters:", count_model_params(model2))

Output shape: torch.Size([1, 2048])
Parameters: {'trainable': 3200576, 'non_trainable': 0, 'total': 3200576}


In [ ]:
configs = get_conv1d_params(in_length=2048, out_length=512, in_channels=4, out_channels=8)
for c in configs:
    print(f"k={c['kernel_size']}, s={c['stride']}, p={c['padding']} | Params: {c['trainable_params']}")
# Pick the standard downsampling layer (k=4, s=2, p=1)
